# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their @id
print('Record Sets:')
for record_set in metadata.record_sets:
    print(f"- @id: {record_set['@id']}, name: {record_set.get('name','N/A')}")
    # List fields for each record set
    print('  Fields:')
    for field in record_set.get('field', []):
        # field can be a dict or @id string depending on how it is referenced
        if isinstance(field, dict):
            print(f"    - @id: {field.get('@id')}, name: {field.get('name')}")
        else:
            print(f"    - @id: {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Define the list of record set @ids. Consult the overview cell above for their actual values.
# For this FAIR^2 dataset, there is typically one main record set containing the primary data—extract all for demonstration.
record_set_ids = [record_set['@id'] for record_set in metadata.record_sets]

# Dictionary to collect DataFrames by record set @id
dfs = {}
for rsid in record_set_ids:
    # The API exposes records by @id
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dfs[rsid] = df
    print(f"Loaded {len(df)} records for record set '{rsid}'. Columns: {list(df.columns)}")

# For illustration, pick the first record set for the next steps
if record_set_ids:
    main_rs = record_set_ids[0]
    display(dfs[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select one numeric field and one grouping field by their `@id` or column name. Refer to the previous overview and extraction for concrete names.

In [ ]:
# Change these field names as appropriate for your record set columns
# For demonstration, use 'mw [da]' as the numeric field (molecular weight, typical in proteomics datasets)
main_df = dfs[main_rs]

# Attempt to auto-detect a numeric column
numeric_candidates = [col for col in main_df.columns if any(s in col.lower() for s in ['mw', 'weight','abundance','count','number','coverage','score'])]
if numeric_candidates:
    numeric_field = numeric_candidates[0]  # For example: 'MW [Da]'
    print(f"Chosen numeric field: {numeric_field}")
else:
    numeric_field = main_df.select_dtypes(include=['float', 'int']).columns[0]
    print(f"Automatically chosen numeric field: {numeric_field}")

# Set a threshold for filtering
if pd.api.types.is_numeric_dtype(main_df[numeric_field]):
    threshold = main_df[numeric_field].mean()  # filter proteins with MW above mean
else:
    # Attempt conversion
    main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
    threshold = main_df[numeric_field].mean()

filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean())
    / filtered_df[numeric_field].std()
)
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a field if available (often 'description', 'accession', or similar)
group_candidates = [col for col in main_df.columns if any(s in col.lower() for s in ['category','group','type','sample'])]
if group_candidates:
    group_field = group_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_'+numeric_field)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())
else:
    print("No suitable group field detected for groupby analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (Molecular Weight)
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# If the grouping exists, plot boxplot by group
if 'group_field' in locals() and group_field in filtered_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we have explored the FAIR^2 dataset 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells' using the `mlcroissant` library. We loaded the Croissant schema, explored the record sets and fields, extracted the main data, performed preliminary filtering and normalization on a numeric attribute (e.g., molecular weight), and visualized the distribution. This workflow serves as a template for further biological or statistical analyses.